Overall population over ticks:

In [ ]:
# --- CONFIGURATION ---
target_run_name = "run one"  # Changed to 'name' for clarity
# ---------------------

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os

# Set up dark theme
plt.style.use('dark_background')
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'legend.fontsize': 14,
})

# Connect to the database
# Using relative path to match your structure
db_path = "data/ecosystem.db"
if not os.path.exists(db_path):
    print(f"❌ Error: Database not found at {db_path}")
else:
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    # 1. Resolve Run Name to internal ID
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM runs WHERE name = ?", (target_run_name,))
    result = cursor.fetchone()

    if result:
        target_run_id = result[0]
        print(f"✅ Analyzing '{target_run_name}' (Internal ID: {target_run_id})")

        # 2. SQL Query using the correct integer ID
        query = f"""
        SELECT 
            t.tick_number, 
            cs.species, 
            COUNT(cs.creature_id) as species_count
        FROM ticks t
        JOIN creature_states cs ON t.id = cs.tick_id
        WHERE t.run_id = {target_run_id}
        GROUP BY t.tick_number, cs.species
        ORDER BY t.tick_number
        """
        
        species_df = pd.read_sql_query(query, conn)

        if not species_df.empty:
            # 3. Pivot the data
            pivot_df = species_df.pivot(index='tick_number', columns='species', values='species_count').fillna(0)
            
            # Match your class names (erf and glooper)
            # Check for case sensitivity based on your creature files
            cols = pivot_df.columns
            prey_col = 'erf' if 'erf' in cols else ('Erf' if 'Erf' in cols else None)
            pred_col = 'glooper' if 'glooper' in cols else ('Glooper' if 'Glooper' in cols else None)

            # 4. Create the Graph
            plt.figure(figsize=(12, 6))
            
            if prey_col:
                plt.plot(pivot_df.index, pivot_df[prey_col], label=f'{prey_col} (Prey)', color='#00FF7F', linewidth=2)
            if pred_col:
                plt.plot(pivot_df.index, pivot_df[pred_col], label=f'{pred_col} (Predator)', color='#FF4500', linewidth=2)

            plt.xlabel('Tick Number')
            plt.ylabel('Population Count')
            plt.title(f'Population Dynamics: {target_run_name}')
            plt.legend()
            plt.grid(True, alpha=0.1)
            plt.tight_layout()
            plt.show()

            # 5. Summary Stats
            print(f"\n--- Statistics for {target_run_name} ---")
            for col in pivot_df.columns:
                final_pop = int(pivot_df[col].iloc[-1])
                print(f"{col.upper()}: Max {int(pivot_df[col].max())} | Final {final_pop}")
        else:
            print(f"No creature state data found for '{target_run_name}'.")
    else:
        print(f"❌ Error: Run named '{target_run_name}' not found in database.")

    conn.close()

male/female population over ticks for erfs:

In [ ]:
# --- CONFIGURATION ---
target_run_name = "run one" 
# ---------------------

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Apply dark theme
plt.style.use('dark_background')
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
})

conn = sqlite3.connect("data/ecosystem.db")

# 1. Resolve Name to ID
cursor = conn.cursor()
cursor.execute("SELECT id FROM runs WHERE name = ?", (target_run_name,))
res = cursor.fetchone()

if not res:
    print(f"❌ Run '{target_run_name}' not found.")
else:
    target_run_id = res[0]

    # 2. Query with the MAX(id) logic to handle resumed duplicates correctly
    query = '''
    SELECT t.tick_number,
           cs.sex,
           COUNT(*) AS count
    FROM creature_states cs
    JOIN (
        SELECT tick_number, MAX(id) AS id
        FROM ticks
        WHERE run_id = ?
        GROUP BY tick_number
    ) t ON cs.tick_id = t.id
    GROUP BY t.tick_number, cs.sex
    ORDER BY t.tick_number, cs.sex
    '''
    
    gender_df = pd.read_sql_query(query, conn, params=(target_run_id,))

    if gender_df.empty:
        print(f"No sex data found for {target_run_name}.")
    else:
        # 3. Pivot and clean labels
        pivot_df = gender_df.pivot(index='tick_number', columns='sex', values='count').fillna(0)
        
        # Rename columns to be pretty, but check if they exist first
        rename_map = {}
        if 'M' in pivot_df.columns: rename_map['M'] = 'Males'
        if 'F' in pivot_df.columns: rename_map['F'] = 'Females'
        pivot_df = pivot_df.rename(columns=rename_map)

        # 4. Plotting
        plt.figure(figsize=(14, 7))
        
        if 'Males' in pivot_df.columns:
            plt.plot(pivot_df.index, pivot_df['Males'], label='Males', color='#3498db', linewidth=3)
        if 'Females' in pivot_df.columns:
            plt.plot(pivot_df.index, pivot_df['Females'], label='Females', color='#e84393', linewidth=3)

        plt.xlabel('Tick Number')
        plt.ylabel('Creature Count')
        plt.title(f'Demographics: {target_run_name}')
        plt.legend()
        plt.grid(True, alpha=0.1)
        plt.tight_layout()
        plt.show()

        # Summary Print
        m_final = int(pivot_df['Males'].iloc[-1]) if 'Males' in pivot_df.columns else 0
        f_final = int(pivot_df['Females'].iloc[-1]) if 'Females' in pivot_df.columns else 0
        print(f"Final Count for '{target_run_name}': {m_final} Males, {f_final} Females")

conn.close()

Geographic heatmap

In [ ]:
import seaborn as sns
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
target_run_name = "run one" 
# ---------------------

plt.style.use('dark_background')
conn = sqlite3.connect("data/ecosystem.db")
conn.row_factory = sqlite3.Row

# 1. Resolve Name to ID
cursor = conn.cursor()
cursor.execute("SELECT id FROM runs WHERE name = ?", (target_run_name,))
res = cursor.fetchone()

if not res:
    print(f"❌ Run '{target_run_name}' not found.")
else:
    target_run_id = res['id']

    # 2. Query to get positions AND species for the last tick
    query = '''
    SELECT cs.pos_x, cs.pos_y, cs.species
    FROM creature_states cs
    JOIN ticks t ON cs.tick_id = t.id
    WHERE t.run_id = ? 
      AND t.tick_number = (SELECT MAX(tick_number) FROM ticks WHERE run_id = ?)
      AND cs.alive = 1
    '''

    try:
        pos_df = pd.read_sql_query(query, conn, params=(target_run_id, target_run_id))

        if not pos_df.empty:
            plt.figure(figsize=(12, 10))
            
            # 3. Create the Species-Specific Density Plot
            # We use 'hue' to separate Erfs and Gloopers
            # 'palette' defines the colors for each species
            sns.kdeplot(
                data=pos_df, 
                x='pos_x', 
                y='pos_y', 
                hue='species',
                fill=True, 
                thresh=0.05, # Removes the very faint outer edges for clarity
                levels=20,    # Number of contour lines
                alpha=0.6,    # Transparency so the layers overlap nicely
                palette={'erf': '#00FF7F', 'glooper': '#FF4500'} # Matching your other graphs
            )

            # 4. Styling
            plt.title(f'Species Concentration Heatmap: {target_run_name}', fontsize=20, pad=20)
            plt.xlabel('World X', fontsize=14)
            plt.ylabel('World Y', fontsize=14)
            
            # Lock the grid to your world bounds (0 to 100)
            plt.xlim(-10, 110) 
            plt.ylim(-10, 110)
            
            plt.grid(True, alpha=0.1)
            plt.tight_layout()
            plt.show()
        else:
            print(f"No living creatures found at the end of '{target_run_name}'.")

    except Exception as e:
        print(f"Error generating heatmap: {e}")

conn.close()